<a href="https://colab.research.google.com/github/Shrutishinha/Machine-Learning-Journey/blob/main/Bayesian_Calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian Calculation — Python Implementation and Explanation

This notebook covers:
1. Bayes' Theorem — formula and intuition
2. A classic worked example: medical test diagnosis
3. Python implementation of Bayes' Theorem (generic function)
4. A second example: spam email classification (Naive Bayes intuition)
5. Visualization of how probabilities update with evidence


## 1. Bayes' Theorem

Bayes' Theorem describes how to update the probability of a hypothesis $H$
given new evidence $E$:

$$ P(H \mid E) = \frac{P(E \mid H) \cdot P(H)}{P(E)} $$

Where:
- $P(H)$ = **Prior** — initial belief about $H$ before seeing evidence
- $P(E \mid H)$ = **Likelihood** — probability of observing evidence $E$ if $H$ is true
- $P(E)$ = **Evidence (marginal probability)** — total probability of observing $E$ across all hypotheses
- $P(H \mid E)$ = **Posterior** — updated belief about $H$ after observing $E$

For a binary hypothesis ($H$ and not $H$), the evidence term expands as:

$$ P(E) = P(E \mid H) P(H) + P(E \mid \neg H) P(\neg H) $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def bayes_theorem(p_h, p_e_given_h, p_e_given_not_h):
    """
    Compute the posterior probability P(H | E) using Bayes' Theorem.

    Parameters
    ----------
    p_h : float
        Prior probability P(H)
    p_e_given_h : float
        Likelihood P(E | H)
    p_e_given_not_h : float
        Likelihood P(E | not H)

    Returns
    -------
    float : Posterior probability P(H | E)
    """
    p_not_h = 1 - p_h
    p_e = (p_e_given_h * p_h) + (p_e_given_not_h * p_not_h)
    p_h_given_e = (p_e_given_h * p_h) / p_e
    return p_h_given_e

## 2. Worked Example: Medical Test Diagnosis

**Scenario:** A disease affects 1% of a population. A test for the disease
is 95% accurate for people who have the disease (sensitivity), and 90%
accurate for people who do not have the disease (specificity, so 10% false
positive rate).

**Question:** If a person tests positive, what is the probability they
actually have the disease?

- $P(H)$ = P(Disease) = 0.01
- $P(E \mid H)$ = P(Positive | Disease) = 0.95
- $P(E \mid \neg H)$ = P(Positive | No Disease) = 0.10 (false positive rate)

In [ ]:
p_disease = 0.01
p_pos_given_disease = 0.95
p_pos_given_no_disease = 0.10

posterior = bayes_theorem(p_disease, p_pos_given_disease, p_pos_given_no_disease)

print(f"P(Disease) [Prior]:                {p_disease:.4f}")
print(f"P(Positive | Disease) [Likelihood]: {p_pos_given_disease:.4f}")
print(f"P(Positive | No Disease):           {p_pos_given_no_disease:.4f}")
print(f"\nP(Disease | Positive) [Posterior]: {posterior:.4f}  ({posterior*100:.2f}%)")

### Interpretation

Even with a positive test result, the probability that the person actually
has the disease is only about **8.8%**. This counter-intuitive result occurs
because the disease is rare (low prior), so most positive results come from
the much larger group of healthy people who get a false positive.

This demonstrates why **base rates (priors) matter enormously** in
probabilistic reasoning — a common pitfall known as the "base rate fallacy.

## 3. Sequential Updating — Multiple Pieces of Evidence

Bayes' Theorem can be applied **iteratively**: the posterior from one piece
of evidence becomes the prior for the next. Suppose the person takes the
same test a second time and tests positive again.

In [ ]:
# Second test, using first posterior as new prior
posterior_2 = bayes_theorem(posterior, p_pos_given_disease, p_pos_given_no_disease)

print(f"Posterior after 1st positive test: {posterior:.4f} ({posterior*100:.2f}%)")
print(f"Posterior after 2nd positive test: {posterior_2:.4f} ({posterior_2*100:.2f}%)")

In [ ]:
# Visualize how belief updates over repeated positive tests
n_tests = 5
beliefs = [p_disease]
current = p_disease

for i in range(n_tests):
    current = bayes_theorem(current, p_pos_given_disease, p_pos_given_no_disease)
    beliefs.append(current)

plt.figure(figsize=(8,5))
plt.plot(range(n_tests + 1), beliefs, marker="o", color="darkred")
plt.xlabel("Number of Positive Test Results")
plt.ylabel("P(Disease | Evidence)")
plt.title("Bayesian Belief Update with Repeated Positive Tests")
plt.grid(True)
plt.show()

for i, b in enumerate(beliefs):
    print(f"After {i} positive test(s): P(Disease) = {b:.4f}")

## 4. AI Application: Naive Bayes Classifier (Spam Detection)

Bayes' Theorem is the foundation of the **Naive Bayes classifier**, widely
used for text classification (e.g., spam filtering). For a document with
words $w_1, w_2, ..., w_n$ and class $C$ (spam or not spam):

$$ P(C \mid w_1,...,w_n) \propto P(C) \prod_{i=1}^{n} P(w_i \mid C) $$

The "naive" assumption is that word occurrences are conditionally
independent given the class. We implement a small example below.

In [ ]:
# Simple Naive Bayes spam classifier example
# Prior probabilities
p_spam = 0.4
p_not_spam = 0.6

# Likelihoods: P(word present | class), estimated from a training set
word_probs = {
    "free":    {"spam": 0.6, "not_spam": 0.05},
    "win":     {"spam": 0.5, "not_spam": 0.02},
    "meeting": {"spam": 0.05, "not_spam": 0.3},
}

def classify_email(words_present):
    """Classify an email as spam or not spam based on word presence using Naive Bayes."""
    score_spam = p_spam
    score_not_spam = p_not_spam

    for word, present in words_present.items():
        p_spam_word = word_probs[word]["spam"] if present else (1 - word_probs[word]["spam"])
        p_not_spam_word = word_probs[word]["not_spam"] if present else (1 - word_probs[word]["not_spam"])
        score_spam *= p_spam_word
        score_not_spam *= p_not_spam_word

    # Normalize to get posterior probabilities
    total = score_spam + score_not_spam
    return score_spam / total, score_not_spam / total

# Example email containing "free" and "win" but not "meeting"
email_words = {"free": True, "win": True, "meeting": False}
p_spam_post, p_not_spam_post = classify_email(email_words)

print(f"Email contains: {[w for w, present in email_words.items() if present]}")
print(f"P(Spam | words)     = {p_spam_post:.4f}")
print(f"P(Not Spam | words) = {p_not_spam_post:.4f}")
print(f"\nClassification: {'SPAM' if p_spam_post > p_not_spam_post else 'NOT SPAM'}")

## 5. Conclusion

- **Bayes' Theorem** provides a principled way to update beliefs (posterior)
  given a prior belief and new evidence (likelihood).
- The medical diagnosis example shows the importance of **prior
  probabilities (base rates)** — a positive test on a rare condition still
  yields a low posterior probability of actually having the condition.
- Repeated evidence can be incorporated through **sequential Bayesian
  updating**, where each posterior becomes the next prior.
- The **Naive Bayes classifier**, a direct AI application of Bayes'
  Theorem, is widely used for text classification tasks such as spam
  detection, sentiment analysis, and document categorization, due to its
  simplicity, speed, and surprisingly strong performance despite the
  independence assumption.

### Visualizing the Spam Classifier Decision Boundary (for 'free' and 'win' words)

To visualize how the Naive Bayes classifier makes decisions, we'll create a heatmap that shows the probability of an email being spam based on the presence or absence of two key words: 'free' and 'win'. For this visualization, we will assume the word 'meeting' is *not* present in the email.

In [4]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Define the words to visualize and fix the state of 'meeting'
words_to_visualize = ['free', 'win']
fixed_word = 'meeting'
fixed_word_presence = False # Assume 'meeting' is not present

# Create a DataFrame to store results
data = []
for free_present in [False, True]:
    for win_present in [False, True]:
        email_words = {
            'free': free_present,
            'win': win_present,
            fixed_word: fixed_word_presence
        }
        p_spam_post, _ = classify_email(email_words)
        data.append({
            'free_present': 'Present' if free_present else 'Absent',
            'win_present': 'Present' if win_present else 'Absent',
            'p_spam': p_spam_post
        })

df_viz = pd.DataFrame(data)

# Reshape for heatmap
heatmap_data = df_viz.pivot(index='win_present', columns='free_present', values='p_spam')

plt.figure(figsize=(8, 6))
sns.heatmap(heatmap_data, annot=True, cmap='viridis', fmt=".3f",
            linewidths=.5, linecolor='black',
            xticklabels=['Free Absent', 'Free Present'],
            yticklabels=['Win Absent', 'Win Present'])
plt.title(f'P(Spam | Free, Win) with {fixed_word} {("Absent" if not fixed_word_presence else "Present")}')
plt.xlabel('Presence of "Free"')
plt.ylabel('Presence of "Win"')
plt.show()

NameError: name 'classify_email' is not defined